# 02 · Feature engineering (Silver & Features)

**Configuración base:**
- Catálogo: `main`
- Schemas: `loterias_raw`, `loterias_bronze`, `loterias_silver`, `loterias_features`
- Volume crudos: `/Volumes/main/loterias_raw/raw_apuestas/apuestas_partitioned/`

In [0]:
CATALOG = "main"
SCHEMA_SILVER   = "loterias_silver"
SCHEMA_FEATURES = "loterias_features"
TABLE_BRONZE  = f"{CATALOG}.loterias_bronze.apuestas_bronze"
TABLE_SILVER  = f"{CATALOG}.{SCHEMA_SILVER}.apuestas_silver"
TABLE_FEATURE = f"{CATALOG}.{SCHEMA_FEATURES}.features_apuestas"

from pyspark.sql import functions as F, Window
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_FEATURES}")

dfb = spark.table(TABLE_BRONZE)
df = (dfb.dropDuplicates(["tx_id"])
         .filter((F.col("monto")>0) & (F.col("monto")<10000))
         .withColumn("fecha", F.to_date("fecha"))
         .withColumn("es_fraude", F.col("es_fraude").cast("int")))
(df.write.format("delta").mode("overwrite").option("overwriteSchema","true")
 .partitionBy("fecha").saveAsTable(TABLE_SILVER))

w_user = Window.partitionBy("user_id")
w_ip   = Window.partitionBy("ip")
# ordenar por segundos unix (mismo tipo que el rango en segundos)
w_t = (Window.partitionBy("user_id")
              .orderBy(F.col("fecha").cast("timestamp").cast("long"))
              .rangeBetween(-7*86400, 0))
df_feat = (df.withColumn("monto_log", F.log1p("monto"))
             .withColumn("freq_usuario", F.count("*").over(w_user))
             .withColumn("urgencia", F.when(F.col("min_antes_cierre")<=10,1).otherwise(0))
             .withColumn("repeticion_ip", F.count("*").over(w_ip))
             .withColumn("prom_monto_usuario", F.avg("monto").over(w_user))
             .withColumn("apuestas_7d_usuario", F.count("tx_id").over(w_t))
             .withColumn("tasa_riesgo_ip", F.avg(F.col("es_fraude").cast("double")).over(w_ip)))
(df_feat.write.format("delta").mode("overwrite").option("overwriteSchema","true")
 .saveAsTable(TABLE_FEATURE))
display(spark.table(TABLE_FEATURE).limit(10))